# Terra + Aqua (combined) / MODIS combined products (MCD)

Example notebook from the [Satellite Data Catalog](https://alessandrosebastianelli.github.io/satcat/satellite.html?folder=modis/modis-terra-aqua-mcd). Each section below runs the Python example of one processing level.

Run the cells top to bottom. Some sources need a free account; the cell that needs credentials asks for them when it runs.

## L3

Level 3 combined: BRDF/albedo and NBAR reflectance (MCD43A1/A3/A4), monthly burned area (MCD64A1), yearly land cover (MCD12Q1), MAIAC aerosol (MCD19A2).

Searches the Microsoft Planetary Computer STAC catalogue (no account needed) for the daily MCD43A4 nadir BRDF-adjusted reflectance tile over Rome and plots band 1 (red).

In [ ]:
%pip install -q pystac-client planetary-computer rioxarray matplotlib

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import planetary_computer
import pystac_client
import rioxarray

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)


def open_overview(href, level=2):
    """Open a COG at reduced resolution (fast), falling back to full resolution."""
    try:
        return rioxarray.open_rasterio(href, overview_level=level, masked=True).squeeze()
    except Exception:
        return rioxarray.open_rasterio(href, masked=True).squeeze()

search = catalog.search(
    collections=["modis-43A4-061"],
    bbox=[12.4, 41.8, 12.6, 42.0],                      # Rome - change to your area
    datetime="2024-07-01/2024-07-05",
)
items = [i for i in search.item_collection() if i.id.startswith("MCD43A4")]
if not items:
    raise SystemExit("No tiles found - widen the dates or the bounding box")
item = items[0]
print(len(items), "tiles;", "using", item.id, item.datetime)
print("assets:", sorted(item.assets))

key = next((k for k in item.assets if "Band1" in k), None)
if key is None:
    raise SystemExit("No asset containing 'Band1' - pick one from the list above")
data = open_overview(item.assets[key].href, level=1) * 0.0001
data.plot(cmap="inferno" if "V" == "K" else "YlGn")
plt.title(item.id + " - " + key + " (NBAR red reflectance)")
plt.show()

## L4

Level 4 combined: LAI and FPAR from both sensors (MCD15A2H 8-day, MCD15A3H 4-day) and other merged model-based products.

Searches NASA Earthdata for the combined MCD15A3H LAI tile over Rome, downloads it (needs a free Earthdata login, asked at run time) and plots LAI. The tile is in sinusoidal projection (shown in pixel coordinates).

In [ ]:
%pip install -q earthaccess pyhdf matplotlib numpy

In [ ]:
from pathlib import Path

import earthaccess
import matplotlib.pyplot as plt
import numpy as np
from pyhdf.SD import SD, SDC

earthaccess.login()   # asks for a (free) NASA Earthdata Login the first time

results = earthaccess.search_data(
    short_name="MCD15A3H",
    version="061",
    bounding_box=(12.2, 41.7, 12.8, 42.1),       # west, south, east, north (central Italy)
    temporal=("2024-07-01", "2024-07-05"),
)
print(len(results), "granules found")
files = earthaccess.download(results[:1], "data")
path = Path(files[0])

hdf = SD(str(path), SDC.READ)
print(sorted(hdf.datasets()))

sds = hdf.select("Lai_500m")
raw = sds[:, :].astype("float32")
raw[raw > 100] = np.nan                           # fill / invalid values
data = raw * 0.1
plt.imshow(data, cmap="inferno")
plt.colorbar(label="LAI (m2/m2)")
plt.title(path.name)
plt.show()